# RevEd — Bulk re-embed to `reved-index-v2` on Colab GPU

Runs the same migration as `scripts/reembed_with_bge.py`, but uses Colab's free T4 GPU so all ~14k chunks finish in ~10–15 min instead of ~4 hours on CPU.

**Before you start**
1. **Runtime → Change runtime type → T4 GPU**. Confirm with the `nvidia-smi` cell below.
2. On your laptop, zip the chunks folder:
   ```
   cd C:\Users\solom\Documents\Startups\RevEd\RedEd\reved-llm-backend\data
   tar -a -c -f chunks_textbooks.zip chunks\textbooks
   ```
   (or right-click the folder → Send to → Compressed (zipped) folder).
3. Also have your existing `migration_v2_status.csv` handy (so Colab skips Biology 1 & 2 that are already indexed).
4. Have your Pinecone API key ready — you'll paste it into a hidden prompt.

**What this does NOT change**
- Your local code stays as-is. This notebook only writes vectors into Pinecone.
- Your live API keeps using the old 384-dim index until you flip `.env`.


## 0. Sanity check — GPU available?

In [ ]:
!nvidia-smi | head -n 15

If you don't see a GPU listed, go to **Runtime → Change runtime type → T4 GPU**, then re-run.

## 1. Install dependencies

In [ ]:
!pip install -q 'sentence-transformers>=2.7,<3.0' 'pinecone>=8.0,<9.0'

## 2. Upload chunks + status CSV

**How to zip on Windows:** open Explorer at `…\reved-llm-backend\data\chunks\`, right-click the `textbooks` folder → **Send to → Compressed (zipped) folder**. This produces `textbooks.zip` with the correct internal structure. Do **not** use PowerShell's `Compress-Archive` — it stores backslashes as literal characters in filenames, which breaks on Linux/Colab. (This cell auto-repairs that case, but Windows Explorer is the clean path.)

When the upload widget appears, select **two files**:
- `textbooks.zip` (or `chunks_textbooks.zip`) — the zipped chunk JSONLs
- `migration_v2_status.csv` — copied from `data/migration_v2_status.csv` (optional but recommended; skips files already indexed)

In [ ]:
import os, shutil, zipfile, glob
from google.colab import files

uploaded = files.upload()
for name in uploaded:
    print(f'Received: {name} ({len(uploaded[name])/1024:.1f} KB)')

# Extract any zip we received
for name in list(uploaded):
    if name.lower().endswith('.zip'):
        with zipfile.ZipFile(name) as zf:
            zf.extractall('.')
        print(f'Extracted {name}')

# Auto-repair PowerShell Compress-Archive zips: backslashes in entry names
# end up as literal characters in filenames instead of path separators.
repaired = 0
for name in list(os.listdir('.')):
    if '\\' in name and os.path.isfile(name):
        target = os.path.join(*name.split('\\'))
        os.makedirs(os.path.dirname(target), exist_ok=True)
        shutil.move(name, target)
        repaired += 1
if repaired:
    print(f'Repaired {repaired} files with literal-backslash names.')

# Find the chunks root. We accept either:
#   chunks/<content_type>/<subject>/*.jsonl       (step 5+ layout)
#   <content_type>/<subject>/*.jsonl              (if user zipped the chunks/ dir)
#   textbook/<subject>/*.jsonl                    (if only one content_type uploaded)
#
# CONTENT_TYPES is the canonical list; we look for any dir that contains at
# least one of these as a direct child.
KNOWN_CONTENT_TYPES = ('textbook', 'teacher_guide', 'syllabus')
candidates = []
for path in glob.glob('**/*', recursive=True):
    if not os.path.isdir(path):
        continue
    children = set(os.listdir(path))
    if children & set(KNOWN_CONTENT_TYPES):
        candidates.append(path)
# Fallback: a directory NAMED after a content type that contains jsonl files.
if not candidates:
    for ct in KNOWN_CONTENT_TYPES:
        for path in glob.glob(f'**/{ct}', recursive=True):
            if os.path.isdir(path) and any(os.scandir(path)):
                candidates.append(os.path.dirname(path) or '.')
                break
        if candidates:
            break

if not candidates:
    raise RuntimeError(
        'Could not find a chunks layout with content_type subdirs '
        f'({KNOWN_CONTENT_TYPES}). Check the zip layout.'
    )
CHUNK_ROOT = candidates[0]
print(f'Chunk root: {CHUNK_ROOT}')
print(f'Content types present: {sorted(set(os.listdir(CHUNK_ROOT)) & set(KNOWN_CONTENT_TYPES))}')

STATUS_CSV = (
    'migration_v2_status.csv'
    if os.path.exists('migration_v2_status.csv') else None
)
print(f'Status CSV: {STATUS_CSV or "(none — will index every file)"}')

## 3. Pinecone API key (hidden prompt)

In [ ]:
import getpass, os
os.environ['PINECONE_API_KEY'] = getpass.getpass('Pinecone API key (pcsk_...): ')

# Step 4 cutover: new namespace, richer metadata.
PINECONE_INDEX_NAME = 'reved-index-v2'
PINECONE_DIMENSION  = 768
PINECONE_METRIC     = 'cosine'
PINECONE_CLOUD      = 'aws'
PINECONE_REGION     = 'us-east-1'
PINECONE_NAMESPACE  = 'kb_v3'          # NEW — replaces 'textbooks' namespace
MODEL_NAME          = 'BAAI/bge-base-en-v1.5'
BATCH_SIZE          = 128
UPSERT_BATCH_SIZE   = 100
PASSAGE_PREFIX      = ''                # bge-base v1.5 needs no passage prefix
INCLUDE_CHUNK_TEXT  = True              # store text in metadata (severs disk dep)
print('Config ready.')

## 4. Load model on GPU

In [ ]:
import torch
from sentence_transformers import SentenceTransformer
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
model = SentenceTransformer(MODEL_NAME, device=device)
print(f'Loaded {MODEL_NAME}, dim={model.get_sentence_embedding_dimension()}')

## 5. Create / verify Pinecone index

In [ ]:
from pinecone import Pinecone, ServerlessSpec
import time

pc = Pinecone(api_key=os.environ['PINECONE_API_KEY'])

if not pc.has_index(PINECONE_INDEX_NAME):
    print(f'Creating index {PINECONE_INDEX_NAME} (dim={PINECONE_DIMENSION}, {PINECONE_METRIC})...')
    pc.create_index(
        name=PINECONE_INDEX_NAME,
        vector_type='dense',
        dimension=PINECONE_DIMENSION,
        metric=PINECONE_METRIC,
        spec=ServerlessSpec(cloud=PINECONE_CLOUD, region=PINECONE_REGION),
        deletion_protection='disabled',
    )

for _ in range(60):
    desc = pc.describe_index(PINECONE_INDEX_NAME)
    status = getattr(desc, 'status', None) or (desc.get('status') if isinstance(desc, dict) else {})
    ready = status.get('ready') if isinstance(status, dict) else getattr(status, 'ready', False)
    if ready:
        break
    time.sleep(2)

index = pc.Index(name=PINECONE_INDEX_NAME)
print(index.describe_index_stats())

## 6. Load resumption state

We accept a row as "already done" only if it matches the current index AND model. That way switching models in the future invalidates the prior runs automatically.

In [ ]:
import csv
from dataclasses import dataclass, asdict
from datetime import datetime, timezone

STATUS_COLUMNS = [
    'chunk_file','subject','source_file','indexing_status','indexed_vector_count',
    'pinecone_index','pinecone_namespace','embedding_model','last_run_at','notes',
]

status_rows = {}
if STATUS_CSV and os.path.exists(STATUS_CSV):
    with open(STATUS_CSV, 'r', encoding='utf-8', newline='') as fh:
        for row in csv.DictReader(fh):
            status_rows[row['chunk_file']] = {c: row.get(c, '') for c in STATUS_COLUMNS}
    print(f'Loaded {len(status_rows)} prior status rows.')
else:
    print('No prior status file — will process every chunk JSONL.')

def save_status():
    out = 'migration_v2_status.csv'
    with open(out, 'w', encoding='utf-8', newline='') as fh:
        writer = csv.DictWriter(fh, fieldnames=STATUS_COLUMNS)
        writer.writeheader()
        for row in sorted(status_rows.values(), key=lambda r: r['chunk_file'].lower()):
            writer.writerow(row)
    return out

## 7. Run the migration

In [ ]:
import json, time
from pathlib import Path

# Walk <content_type>/<subject>/*.jsonl across all content types under the chunk root.
# Defensively skip any *_v1_backup directories so historical chunks don't get
# re-embedded into the live namespace.
all_jsonl = sorted(Path(CHUNK_ROOT).rglob('*.jsonl'))
chunk_files = [p for p in all_jsonl if 'v1_backup' not in str(p)]
skipped_backup = len(all_jsonl) - len(chunk_files)
print(f'Found {len(chunk_files)} chunk file(s); skipped {skipped_backup} backup file(s).')

def iter_records(path):
    with open(path, 'r', encoding='utf-8') as fh:
        for line in fh:
            line = line.strip()
            if line:
                yield json.loads(line)

# Source-driven visibility is set at chunk-write time on disk; the notebook
# just mirrors what's already in the record.
def build_metadata(record: dict) -> dict:
    md = {
        'source_file':  record['source_file'],
        'source_path':  record['source_path'],
        'subject':      record['subject'],
        'content_type': record['content_type'],
        'chunk_index':  int(record['chunk_index']),
        'total_chunks': int(record['total_chunks']),
        'document_id':  record['document_id'],
        'char_count':   int(record['char_count']),
    }
    for key, cast in (
        ('token_count',   int),
        ('chapter',       str),
        ('section',       str),
        ('topic',         str),
        ('level',         str),
        ('board',         str),
        ('chunk_type',    str),
        ('visibility',    str),
        ('content_hash',  str),
    ):
        val = record.get(key)
        if val not in (None, '', []):
            md[key] = cast(val)
    tags = record.get('tags')
    if tags:
        md['tags'] = list(tags)
    if INCLUDE_CHUNK_TEXT:
        md['text'] = record['text']
    return md

def chunk_file_relpath(chunk_file: Path) -> str:
    """Stable identifier for the status CSV: data/chunks/<content_type>/<subject>/<file>.jsonl"""
    try:
        rel = chunk_file.relative_to(CHUNK_ROOT)
    except ValueError:
        rel = Path(chunk_file.name)
    return f'data/chunks/{rel.as_posix()}'

total_indexed = 0
for chunk_file in chunk_files:
    rel_posix = chunk_file_relpath(chunk_file)
    prior = status_rows.get(rel_posix)
    if (
        prior
        and prior.get('indexing_status') == 'indexed'
        and prior.get('pinecone_index') == PINECONE_INDEX_NAME
        and prior.get('pinecone_namespace') == PINECONE_NAMESPACE
        and prior.get('embedding_model') == MODEL_NAME
    ):
        print(f'SKIP  {rel_posix}  ({prior.get("indexed_vector_count")} already indexed)')
        total_indexed += int(prior.get('indexed_vector_count') or 0)
        continue

    records = list(iter_records(chunk_file))
    if not records:
        print(f'EMPTY {rel_posix}')
        continue

    subject = records[0].get('subject', chunk_file.parent.name.lower())
    content_type = records[0].get('content_type', chunk_file.parent.parent.name.lower())
    source_file = records[0].get('source_file', f'{chunk_file.stem}.txt')

    t0 = time.time()
    indexed = 0
    for start in range(0, len(records), BATCH_SIZE):
        batch = records[start:start + BATCH_SIZE]
        texts = [(PASSAGE_PREFIX + r['text']) if PASSAGE_PREFIX else r['text'] for r in batch]
        vectors = model.encode(
            texts,
            batch_size=BATCH_SIZE,
            normalize_embeddings=True,
            convert_to_numpy=True,
            show_progress_bar=False,
        )

        payload = [
            {'id': r['chunk_id'], 'values': vec.tolist(), 'metadata': build_metadata(r)}
            for r, vec in zip(batch, vectors)
        ]
        for u_start in range(0, len(payload), UPSERT_BATCH_SIZE):
            response = index.upsert(
                vectors=payload[u_start:u_start + UPSERT_BATCH_SIZE],
                namespace=PINECONE_NAMESPACE,
            )
            count = getattr(response, 'upserted_count', None)
            if count is None and isinstance(response, dict):
                count = response.get('upserted_count') or response.get('upsertedCount') or 0
            indexed += int(count or 0)

    elapsed = time.time() - t0
    print(f'DONE  {rel_posix}  chunks={len(records)} indexed={indexed} ({elapsed:.1f}s)')
    status_rows[rel_posix] = {
        'chunk_file': rel_posix,
        'subject': subject,
        'source_file': source_file,
        'indexing_status': 'indexed',
        'indexed_vector_count': str(indexed),
        'pinecone_index': PINECONE_INDEX_NAME,
        'pinecone_namespace': PINECONE_NAMESPACE,
        'embedding_model': MODEL_NAME,
        'last_run_at': datetime.now(timezone.utc).isoformat(timespec='seconds'),
        'notes': f'Indexed {indexed} chunk vector(s); content_type={content_type}.',
    }
    save_status()
    total_indexed += indexed

print(f'\nALL DONE. Total vectors confirmed across all files: {total_indexed}')
print(index.describe_index_stats())

## 8. Download the updated status CSV (replaces the local copy)

In [ ]:
from google.colab import files
files.download('migration_v2_status.csv')

## 9. Spot-check retrieval (sanity test)

In [ ]:
QUERY_PREFIX = 'Represent this sentence for searching relevant passages: '

def search(q, top_k=5, subject=None):
    qv = model.encode([QUERY_PREFIX + q], normalize_embeddings=True, convert_to_numpy=True)[0].tolist()
    flt = {'subject': {'$eq': subject}} if subject else None
    return index.query(
        namespace=PINECONE_NAMESPACE,
        vector=qv,
        top_k=top_k,
        filter=flt,
        include_metadata=True,
    )

for q in [
    'Explain Newton\'s second law and give the formula relating force, mass, and acceleration.',
    'What is the difference between mitosis and meiosis?',
    'Define an ionic bond and give two examples.',
]:
    print(f'\nQ: {q}')
    res = search(q, top_k=3)
    for m in res.matches:
        meta = m.metadata or {}
        print(f'  {m.score:.3f}  [{meta.get("subject")}] {meta.get("source_file")} chunk #{meta.get("chunk_index")}  id={m.id}')

---
When this notebook finishes:
1. Download `migration_v2_status.csv` (cell 8) and drop it into `data/migration_v2_status.csv` in the repo.
2. Vectors are live in Pinecone `reved-index-v2` namespace **`kb_v3`** (new in step 4).
3. To switch the live API over to the new namespace, update `.env`:
   ```
   PINECONE_INDEX_NAME=reved-index-v2
   PINECONE_DIMENSION=768
   HF_EMBEDDING_MODEL=BAAI/bge-base-en-v1.5
   EMBEDDING_BACKEND=local
   PINECONE_NAMESPACE=kb_v3              # ← was 'textbooks'
   PINECONE_INCLUDE_CHUNK_TEXT=true
   ```
   ...and restart the server. Old `textbooks` namespace stays untouched until you delete it from the Pinecone console — easy rollback.